# 🛡️ NETRA — Intelligent Phishing Detection System
## Tier-1 Model Training Notebook

**Project**: NETRA — The AI Eye Against Phishing  
**Phase**: 1 — Data Pipeline + Tier-1 Classical ML  
**Models**: Logistic Regression, Linear SVM, Random Forest  
**Target**: Phishing Recall ≥ 0.90, FPR ≤ 0.05, F1 ≥ 0.85  

---
### 📋 Before Running This Notebook
1. Upload your datasets to Google Drive under `My Drive/NETRA/data/raw/`  
2. Upload the entire `NETRA` project folder to Google Drive  
3. Run cells in order — each cell depends on the previous  

### Dataset files expected in `/content/drive/MyDrive/NETRA/data/raw/`
| File | Source | Label |
|------|--------|-------|
| `enron_*.mbox` | Enron email corpus | 0 (Legit) |
| `nazario_phishing.csv` | Nazario corpus | 1 (Phishing) |
| `easy_ham.mbox`, `spam_2.mbox` | SpamAssassin | 0/1 |
| `phishtank_online_valid.csv` | PhishTank | 1 (Phishing) |
| `openphish_feed.txt` | OpenPhish | 1 (Phishing) |

## Cell 1: Install Dependencies
Install all required Python packages. Colab already has most; this ensures exact versions.

In [ ]:
# Cell 1: Install dependencies
print('Installing NETRA dependencies...')

!pip install -q \
    scikit-learn==1.5.2 \
    pandas==2.2.3 \
    numpy==1.26.4 \
    joblib==1.4.2 \
    imbalanced-learn==0.12.3 \
    matplotlib==3.9.2 \
    seaborn==0.13.2 \
    nltk==3.9.1 \
    fastapi==0.115.0 \
    httpx==0.27.2

print('✅ Dependencies installed.')

## Cell 2: Mount Google Drive and Set Paths
Mount your Google Drive and configure the project paths.

In [ ]:
# Cell 2: Mount Google Drive
import os
import sys
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ─── Configure your project path here ───────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/NETRA')
# ─────────────────────────────────────────────────────────────────────────

DATA_RAW     = DRIVE_ROOT / 'data' / 'raw'
DATA_PROC    = DRIVE_ROOT / 'data' / 'processed'
MODELS_DIR   = DRIVE_ROOT / 'ml' / 'models'
NOTEBOOKS_DIR = DRIVE_ROOT / 'notebooks'

# Create directories if they don't exist
for d in [DATA_RAW, DATA_PROC, MODELS_DIR, NOTEBOOKS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Add project root to Python path so imports work
if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, str(DRIVE_ROOT))

print(f'📁 Project root : {DRIVE_ROOT}')
print(f'📁 Raw data     : {DATA_RAW}')
print(f'📁 Models dir   : {MODELS_DIR}')

# List raw data files
raw_files = list(DATA_RAW.rglob('*'))
print(f'\n📋 Found {len(raw_files)} items in data/raw/')
for f in raw_files[:20]:
    print(f'   {f.name}')
if not raw_files:
    print('⚠️  No raw data files found. Upload datasets to:', DATA_RAW)

## Cell 3: Run Data Pipeline
Parse all raw datasets, deduplicate, and write `data/processed/unified.csv`.

This step handles: Enron mbox → label 0, Nazario CSV → label 1, SpamAssassin mbox, PhishTank CSV, OpenPhish feed.

In [ ]:
# Cell 3: Run Data Pipeline
import subprocess
import sys

pipeline_script = DRIVE_ROOT / 'ml' / 'data_pipeline.py'
unified_csv     = DATA_PROC / 'unified.csv'

print('🔄 Running data pipeline...')
result = subprocess.run(
    [
        sys.executable,
        str(pipeline_script),
        '--raw-dir', str(DATA_RAW),
        '--output',  str(unified_csv),
    ],
    capture_output=True, text=True
)

print(result.stdout)
if result.returncode != 0:
    print('❌ Pipeline errors:')
    print(result.stderr)
else:
    print('✅ Pipeline complete!')

# Preview the output
import pandas as pd
if unified_csv.exists():
    df = pd.read_csv(unified_csv)
    print(f'\n📊 unified.csv: {len(df):,} records')
    print(df['split'].value_counts().to_string())
    print(f"\nLabel distribution:\n{df['label'].value_counts().to_string()}")
    print('\nSample rows:')
    display(df[['record_id','source','label','split','subject']].head(5))

## Cell 4: Train Tier-1 Models
Train Logistic Regression, Linear SVM, and Random Forest on the training split.
Evaluates each on the validation split and saves the best model to `ml/models/`.

**Target Metrics**: Phishing Recall ≥ 0.90 · FPR ≤ 0.05 · F1 ≥ 0.85

In [ ]:
# Cell 4: Train Tier-1 Models
import subprocess
import sys

train_script = DRIVE_ROOT / 'ml' / 'train.py'

print('🚀 Starting Tier-1 training...')
print('(This may take 5–20 minutes on Colab free tier for Random Forest)')
print('-' * 60)

result = subprocess.run(
    [
        sys.executable,
        str(train_script),
        '--data', str(unified_csv),
    ],
    capture_output=True, text=True
)

print(result.stdout)
if result.returncode != 0:
    print('❌ Training errors:')
    print(result.stderr)
else:
    print('✅ Training complete!')

# Load and display training metrics
import json
metrics_path = DRIVE_ROOT / 'ml' / 'models' / 'training_metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print(f"\n🏆 Best model: {metrics['best_model']}")
    print('\n📈 All model results:')
    results_df = pd.DataFrame(metrics['all_results'])
    display(results_df.set_index('model'))

## Cell 5: Evaluation Plots
Confusion matrix + classification report + metric comparison bar chart for the mentor demo.

In [ ]:
# Cell 5: Evaluation Plots
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, precision_recall_curve, average_precision_score
)
from scipy.sparse import hstack, csr_matrix

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# --- Load model and data ---
model = joblib.load(MODELS_DIR / 'tier1_model.pkl')
text_extractor = joblib.load(MODELS_DIR / 'tfidf_vectorizer.pkl')

df = pd.read_csv(unified_csv)
df_val = df[df['split'] == 'val'].reset_index(drop=True)

# Re-assemble feature matrix for validation set
from ml.features.url_features import extract as url_extract, URL_FEATURE_NAMES
from ml.features.header_features import extract as header_extract, HEADER_FEATURE_NAMES

X_text = text_extractor.transform(df_val)

url_rows = []
for urls_json in df_val['urls'].fillna('[]'):
    try:
        urls = json.loads(urls_json)
    except:
        urls = []
    url_rows.append([url_extract(urls)[k] for k in URL_FEATURE_NAMES])
X_url = csr_matrix(np.array(url_rows, dtype=np.float32))

header_rows = []
for _, row in df_val[['headers_available','sender','reply_to']].iterrows():
    try:
        headers = json.loads(row['headers_available'] or '{}')
    except:
        headers = {}
    feat = header_extract(headers_dict=headers, sender=row['sender'] or '', reply_to=row['reply_to'] or '')
    header_rows.append([feat[k] for k in HEADER_FEATURE_NAMES])
X_header = csr_matrix(np.array(header_rows, dtype=np.float32))

X_val = hstack([X_text, X_url, X_header])
y_val = df_val['label'].values

y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('NETRA Tier-1 — Validation Set Evaluation', fontsize=15, fontweight='bold')

# --- Plot 1: Confusion Matrix ---
cm = confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Phishing'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontweight='bold')

# --- Plot 2: Metric Comparison Bar Chart ---
with open(metrics_path) as f:
    metrics_data = json.load(f)
results_df = pd.DataFrame(metrics_data['all_results'])
metric_cols = ['accuracy', 'precision', 'recall', 'f1']
results_df[metric_cols].plot(
    kind='bar', ax=axes[1],
    xlabel='', rot=20, ylim=(0, 1)
)
axes[1].set_xticklabels([m.split()[0] for m in results_df['model']], rotation=15)
axes[1].set_title('Model Comparison', fontweight='bold')
axes[1].legend(loc='lower right', fontsize=8)
axes[1].axhline(0.90, color='red', linestyle='--', alpha=0.5, label='Recall target')
axes[1].axhline(0.85, color='orange', linestyle='--', alpha=0.5, label='F1 threshold')

# --- Plot 3: Precision-Recall Curve ---
precision, recall, _ = precision_recall_curve(y_val, y_prob)
pr_auc = average_precision_score(y_val, y_prob)
axes[2].plot(recall, precision, lw=2, color='steelblue', label=f'PR-AUC = {pr_auc:.3f}')
axes[2].fill_between(recall, precision, alpha=0.1, color='steelblue')
axes[2].axhline(0.90, color='red', linestyle='--', alpha=0.5, label='Recall=0.90')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall Curve', fontweight='bold')
axes[2].legend()
axes[2].set_xlim([0, 1])
axes[2].set_ylim([0, 1])

plt.tight_layout()
plot_path = NOTEBOOKS_DIR / 'eval_plots.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plots saved to {plot_path}')

# --- Classification Report ---
print('\n📋 Classification Report:')
print(classification_report(y_val, y_pred, target_names=['Legitimate', 'Phishing']))

## Cell 6: Save Model Artifacts to Google Drive
Verify model files are saved and display their paths for download.

In [ ]:
# Cell 6: Verify & report saved artifacts
from pathlib import Path
import os

artifacts = [
    MODELS_DIR / 'tier1_model.pkl',
    MODELS_DIR / 'tfidf_vectorizer.pkl',
    MODELS_DIR / 'training_metrics.json',
    unified_csv,
]

print('📦 NETRA Model Artifacts:')
print('=' * 55)
all_ok = True
for artifact in artifacts:
    if artifact.exists():
        size_mb = artifact.stat().st_size / (1024 * 1024)
        print(f'  ✅ {artifact.name:<35} {size_mb:.2f} MB')
    else:
        print(f'  ❌ {artifact.name:<35} NOT FOUND')
        all_ok = False
print('=' * 55)

if all_ok:
    print()
    print('🎉 All artifacts saved to Google Drive!')
    print()
    print('📥 Next steps — on your LOCAL machine:')
    print('  1. Download tier1_model.pkl and tfidf_vectorizer.pkl from:')
    print(f'     {MODELS_DIR}')
    print('  2. Place them in: ml/models/')
    print('  3. Start the API server:')
    print('     uvicorn api.main:app --host 127.0.0.1 --port 8000')
    print('  4. Test with curl:')
    print(r'''     curl -X POST http://127.0.0.1:8000/predict \''')
    print(r'''       -H "Content-Type: application/json" \''')
    print(r'''       -d \'{{"body_text": "Your account is suspended. Verify immediately.", "urls": ["http://phish.tk/login"]}}\''')
else:
    print('⚠️  Some artifacts are missing. Re-run cells 4–5.')